In [1]:
import os 
from prepare_data import create_video_frames_df
import random
import numpy as np
import torch
import torchvision.transforms as T
import pandas as pd
from train import train, train_v2
from eval import eval
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from dataset import VideoYoloDataset, TemporalWaggleCollator
from model import R2Plus1D_YOLO
from loss import WaggleDetectionLoss
from augmentation import WaggleAugmentations
from aug_vis import demo_visualization
import torch.nn  as nn
from utils.data_utils import fix_dataframe_with_video_lengths, find_overlapping_rows, preds_to_df, save_preds_to_csv
import datetime
from torch.utils.tensorboard import SummaryWriter
from utils.eval_utils import get_preds_gt, yolo_to_img_space, yolo_to_img_space_gt, get_eval_metrics
import argparse
from utils.vis_utils import reverse_transform_batch, save_frames
from utils.nms import batch_postprocess_predictions
from utils.video_utils import frames_to_video
from utils.draw_utils import  draw_waggle, draw_waggle_batch, draw_waggle_batch_union
from utils.model_utils import load_pretrained_model


### Set Seeds

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Define Config

In [3]:
batch_size = 16
num_epochs = 1
max_detections_per_cell = 1
grid_size = 25
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = os.path.join("logs", current_time)
writer = SummaryWriter(log_dir=log_dir)


### Load Data

In [4]:
csv_path = os.path.join(os.curdir,"/home/prajna/multiscale_wdd/data/annotations","extended_windows_data_pos_full_ids.csv")
data = pd.read_csv(csv_path)

video_frames_dict = create_video_frames_df(data["video_name"].unique())
data = fix_dataframe_with_video_lengths(data, video_frames_dict)
data = data.sample(frac=1).reset_index(drop=True)

transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
    ])

total_len = len(data)
train_len = int(0.8 * total_len)

train_indices = list(range(train_len))
test_indices = list(range(train_len, total_len))

train_df = data.iloc[train_indices].reset_index(drop=True)
test_df = data.iloc[test_indices].reset_index(drop=True)

# sort by video name and start frames to recover chronological/sequential order
train_df = train_df.sort_values(['video_name', 'start_frame']).reset_index(drop=True)
test_df = test_df.sort_values(['video_name', 'start_frame']).reset_index(drop=True)

# filter out non-overlapping windows
test_df = find_overlapping_rows(test_df)

test_dataset = VideoYoloDataset(
    test_df,
    video_frames_dict, 
    transforms,
    width=224,
    height=224,
    clip_len=16,
    grid_size=25,
    max_detections_per_cell=1,
    num_classes=1,
    augment=False, 
    )
    
collator = TemporalWaggleCollator()

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=collator, 
    shuffle=False, 
    num_workers=4,
    pin_memory=True
    )


### Initalise Model

In [5]:
model = load_pretrained_model('/home/deniz/waggle_dance_detector/ckpt/latest.pth', device)
yolocriteria = WaggleDetectionLoss()

/home/deniz/anaconda3/envs/waggle_dance/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/deniz/anaconda3/envs/waggle_dance/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R2Plus1D_18_Weights.KINETICS400_V1`. You can also use `weights=R2Plus1D_18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading pretrained weights from /home/deniz/waggle_dance_detector/ckpt/latest.pth
Loaded model from state dict


### Eval/Train Loop

In [ ]:
for epoch in range(num_epochs):
    test_loss = eval(model, test_loader, yolocriteria, device, epoch, writer)
    # Its not possible to fit all training or test frames onto cpu for visualisations
    # batch_idx_for_frames is set to 0 indicating that it will index into the first batch of the entire data loader and store the frames in there
    # if batch_size is set to 16, that means we have 16*window_size frames in our case 16 * 16, each individual batch represents a single waggle dance event of 16 frames
    # with 16 batches that is 16 * 16
    test_preds_raw, test_gt_raw, test_all_starts, test_all_ends, _ , test_frames = get_preds_gt(model, test_loader, device, return_frames=True, batch_idx_for_frames=0)
    test_frames = reverse_transform_batch(test_frames, original_size=(960,540))

    # visualize fetched test frames as a video if you want
    frames_to_video(test_frames, output_name= 'data_loader_batches_0')
    
    # Transform yolo coordinates onto image domain for both gt and predicted values
    test_gts = yolo_to_img_space_gt(test_gt_raw, all_starts=test_all_starts, all_ends=test_all_ends, window_size = 16, original_size=(960,540))
    test_preds  = yolo_to_img_space(test_preds_raw, all_starts=test_all_starts, all_ends=test_all_ends, confidence_threshold=0.95, window_size = 16, original_size=(960,540))

    # Draw gt and predictions onto frames and saves as video
    # Note: This shows each 16-frame window independently, so frames repeat at window intersections
    draw_waggle_batch(
        all_frames=test_frames,
        all_detections=test_preds, 
        all_ground_truths=test_gts,
        all_start_frame_idxs=test_all_starts
    )

    # This creates a continuous timeline without repeating frames
    # We use [:16] because test_frames only contains the first 16 sequences (batch #0),
    # while test_preds/test_gts contain predictions for all 964 sequences in the test set
    draw_waggle_batch_union(
        all_frames=test_frames,
        all_detections=test_preds[:16],  # Only first 16 sequences (matches our saved frames)
        all_ground_truths=test_gts[:16], # Only first 16 sequences  
        all_start_frame_idxs=test_all_starts[:16]  # Only first 16 start indices
    )

    # Post Process all predictions
    # You can try different strategies if you want but, only cluster_consolidate is important for our purpose. 
    # 'cluster_consolidate' is DBSCAN across spatial and temporal aspect. 4 strategies are supported you can have a look if you want. 
    # mode supporst mean or median, meaning it will compute a point that summarises a cluster based on median or mean summary. 
    post_test_preds = batch_postprocess_predictions(test_preds, spatial_threshold=40,temporal_threshold=7, confidence_threshold=0.95, strategy='cluster_consolidate', mode='median')

    # Can postprocess entire predictions no need for limit to 16 sequences, its only needed when we visualise
    save_preds_to_csv(test_preds, f'raw_predictions_epoch_{epoch}.csv', 'raw', 'outputs')
    save_preds_to_csv(post_test_preds, f'postprocessed_predictions_epoch_{epoch}.csv', 'postprocessed', 'outputs')

    # Print unique clusters identifies
    unique_clusters_all = len({det['cluster_id'] for seq in post_test_preds for det in seq})
    unique_clusters_frames = len({det['cluster_id'] for seq in post_test_preds[:16] for det in seq})

    print('Number of clusterns - Entire data loader:', unique_clusters_all)
    print('Number of clusterns - Frames used for visualisation:', unique_clusters_frames)

    # Visualise post processed results
    draw_waggle_batch_union(
        all_frames=test_frames,
        all_detections=post_test_preds[:16], 
        all_ground_truths=test_gts[:16],
        all_start_frame_idxs=test_all_starts,
        file_name='dbscan',
    )

    # Compute eval metrics
    test_metrics = get_eval_metrics(test_preds, test_gts, pos_threshold=30)
    post_test_metrics = get_eval_metrics(post_test_preds, test_gts, pos_threshold=30)

    print(f"Before Post-Processing: Preds={test_metrics['total_predictions']}, GTs={test_metrics['total_ground_truths']}, Prec={test_metrics['precision']:.3f}, Rec={test_metrics['recall']:.3f}, F1={test_metrics['f1_score']:.3f}, PosErr={test_metrics['mean_position_error']:.1f}px, AngErr={test_metrics['mean_angular_error']:.1f}°")
    print(f"After Post-Processing: Preds={post_test_metrics['total_predictions']}, GTs={post_test_metrics['total_ground_truths']}, Prec={post_test_metrics['precision']:.3f}, Rec={post_test_metrics['recall']:.3f}, F1={post_test_metrics['f1_score']:.3f}, PosErr={post_test_metrics['mean_position_error']:.1f}px, AngErr={post_test_metrics['mean_angular_error']:.1f}°")

print('Eval on single checkpoint complete.')
writer.close()


Validation Results - Epoch 1:
Total Loss: 1.5910
Object Loss: 0.0209
No Object Loss: 0.2241
Position Loss: 0.0729
Direction Loss: 0.0627
Temporal Loss: 0.0645


Video saved to vids/data_loader_batches_0.mp4
Total frames: 256
Created MP4 with 256 frames: vids/all_batches_waggle_detection.mp4
Created Union MP4 with 183 unique frames: vids/union_waggle_detection.mp4
Saved 14692 raw predictions to outputs/raw_predictions_epoch_0.csv
Saved 2497 postprocessed predictions to outputs/postprocessed_predictions_epoch_0.csv
Number of clusterns - Entire data loader: 9
Number of clusterns - Frames used for visualisation: 3
Created Union MP4 with 183 unique frames: vids/union_waggle_detection_dbscan.mp4
Before Post-Processing: Preds=14692, GTs=964, Prec=0.050, Rec=0.769, F1=0.095, PosErr=7.3px, AngErr=14.7°
After Post-Processing: Preds=2497, GTs=964, Prec=0.191, Rec=0.495, F1=0.276, PosErr=11.7px, AngErr=16.6°
Eval on single checkpoint complete.
